# 03 — Budget Optimizer & Saturation Curves

Goal: produce saturation curves and an optimal impression/volume allocation table.

**Input:** `data/processed/trace.nc`, `data/processed/clean.parquet`  
**Output:** `data/processed/roas_curves.parquet`, `data/processed/optimal_allocation.parquet`

> Run `02_model.ipynb` first.

## Channel split

| Role | Channels | Why |
|------|----------|-----|
| **Optimised** | `Paid_Views`, `Google_Impressions`, `Email_Impressions`, `Facebook_Impressions` | We directly control volume/spend |
| **Fixed (held constant)** | `Organic_Views`, `Affiliate_Impressions` | Not directly purchasable — frozen at current mean |

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

from src.mmm import BayesianMMM, BudgetOptimizer
from src.mmm.optimize import CONTROLLABLE_COLS, FIXED_COLS

PROC = Path('../data/processed')
TARGET_COL   = 'Sales'
ALL_CHANNELS = [
    'Paid_Views',
    'Google_Impressions',
    'Email_Impressions',
    'Facebook_Impressions',
    'Affiliate_Impressions',
]

print('Controllable (will be optimised):', CONTROLLABLE_COLS)
print('Fixed (held constant):           ', FIXED_COLS)

## 1. Load fitted model and data

In [ ]:
mmm = BayesianMMM.load(
    str(PROC / 'trace.nc'),
    channel_cols=ALL_CHANNELS,
    control_cols=['Organic_Views'],
    date_col='date',
    target_col=TARGET_COL,
)

# Optimizer only exposes CONTROLLABLE_COLS for optimisation
opt = BudgetOptimizer(mmm, controllable_cols=CONTROLLABLE_COLS)

df = pd.read_parquet(PROC / 'clean.parquet')
print('Model and data loaded.')
print('Optimiser controllable channels:', opt.controllable_cols)

## 2. Current mean volumes — controllable vs fixed

In [ ]:
print('--- CONTROLLABLE (optimisable) ---')
controllable_means = df[CONTROLLABLE_COLS].mean()
for ch, v in controllable_means.items():
    print(f'  {ch:<30} {v:>12,.1f}')

print('\n--- FIXED (held constant in optimiser) ---')
for ch in FIXED_COLS:
    if ch in df.columns:
        print(f'  {ch:<30} {df[ch].mean():>12,.1f}  [not optimised]')

total_controllable_budget = float(controllable_means.sum())
print(f'\nTotal controllable volume: {total_controllable_budget:,.1f}')

## 3. Saturation curves — controllable channels only

In [ ]:
ncols = 2
nrows = -(-len(CONTROLLABLE_COLS) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 4))

all_curves = []
for ax, ch in zip(axes.flat, CONTROLLABLE_COLS):
    mean_vol = df[ch].mean()
    vol_range = np.linspace(0, mean_vol * 3, 300)
    curve = opt.roas_curve(ch, vol_range)
    curve['channel'] = ch
    all_curves.append(curve)

    ax.plot(curve['volume'], curve['response'], linewidth=2)
    ax.axvline(mean_vol, color='red', linestyle='--', alpha=0.7, label=f'Mean ({mean_vol:,.0f})')
    ax.set_title(ch, fontsize=11)
    ax.set_xlabel('Impressions / Views')
    ax.set_ylabel('Normalised Response')
    ax.legend(fontsize=9)

for ax in axes.flat[len(CONTROLLABLE_COLS):]:
    ax.set_visible(False)

plt.suptitle('Saturation Curves — Controllable Channels Only', fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(PROC / 'fig_roas_curves.png', bbox_inches='tight')

roas_df = pd.concat(all_curves, ignore_index=True)
roas_df.to_parquet(PROC / 'roas_curves.parquet', index=False)
print(f'Saved {len(roas_df):,} rows → data/processed/roas_curves.parquet')

## 4. Saturation level at current volume

How close is each channel to its saturation ceiling? High % = diminishing returns, consider shifting budget away.

In [ ]:
print(f'{'Channel':<30} {'Saturation at mean vol':>22} {'Marginal response / 1k':>24}')
print('-' * 78)
for ch in CONTROLLABLE_COLS:
    mean_vol = df[ch].mean()
    vol_range = np.linspace(0, mean_vol * 3, 3000)
    curve = opt.roas_curve(ch, vol_range)
    idx = np.searchsorted(curve['volume'], mean_vol)
    sat = curve['response'].iloc[idx] * 100
    if 0 < idx < len(curve) - 1:
        marginal = (
            (curve['response'].iloc[idx+1] - curve['response'].iloc[idx-1])
            / (curve['volume'].iloc[idx+1] - curve['volume'].iloc[idx-1])
        ) * 1000
    else:
        marginal = float('nan')
    print(f'{ch:<30} {sat:>20.1f}%  {marginal:>22.5f}')

## 5. Budget optimisation — controllable channels only

In [ ]:
current_volumes = df[CONTROLLABLE_COLS].mean().values

result = opt.optimize(
    total_budget=total_controllable_budget,
    lower_bounds=[0.0] * len(CONTROLLABLE_COLS),
    upper_bounds=[total_controllable_budget] * len(CONTROLLABLE_COLS),
    current_volumes=current_volumes,
)
print(result.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: current vs optimal
x = np.arange(len(CONTROLLABLE_COLS))
width = 0.35
axes[0].bar(x - width/2, result['current_volume'],  width, label='Current')
axes[0].bar(x + width/2, result['optimal_volume'], width, label='Optimal')
axes[0].set_xticks(x)
axes[0].set_xticklabels([c.replace('_', '\n') for c in CONTROLLABLE_COLS], fontsize=9)
axes[0].set_ylabel('Volume (impressions / views)')
axes[0].set_title('Current vs Optimal Volume')
axes[0].legend()

# Delta bar chart
colors = ['#2ca02c' if d >= 0 else '#d62728' for d in result['delta']]
axes[1].bar(x, result['delta'], color=colors)
axes[1].set_xticks(x)
axes[1].set_xticklabels([c.replace('_', '\n') for c in CONTROLLABLE_COLS], fontsize=9)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_ylabel('Change in volume')
axes[1].set_title('Recommended Reallocation (green = increase, red = decrease)')

plt.suptitle('Budget Optimisation — Controllable Channels', fontsize=12, y=1.01)
plt.tight_layout()
fig.savefig(PROC / 'fig_optimal_allocation.png', bbox_inches='tight')

result.to_parquet(PROC / 'optimal_allocation.parquet', index=False)
print('Saved → data/processed/optimal_allocation.parquet')